In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def dos_picos(image):
    hist = cv2.calcHist([image], [0], None, [256], [0, 256]) #hago uso de histograma
    #seleccionamos los 2 picos 
    pico=np.array(hist)
    pico=pico.flatten()
    pico_dif=np.diff(pico) #diferencia entre los tipos
    primer_pico=np.argmax(pico_dif[:128]) #primer pico
    segundo_pico=np.argmax(pico_dif[128:])+128 #segundo pico

    umbral= int((primer_pico + segundo_pico) / 2) #umbral entre los 2 picos
    return umbral

imagen=cv2.imread('imagenumbra.jpeg', 0)
umbral_2_picos=dos_picos(imagen)

_, nueva = cv2.threshold(imagen, umbral_2_picos, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(imagen, cmap='gray')
plt.title('Imagen inicial')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(nueva, cmap='gray')
plt.title(f'Imagen umbralizada ')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def isodata(image):
    umbral = 128
    while True:
        grupo1 = image[image <= umbral]  # grupo donde el umbral es menor o igual
        grupo2 = image[image > umbral]   # grupo donde el umbral es mayor

        media1 = np.mean(grupo1) if grupo1.size > 0 else 0
        media2 = np.mean(grupo2) if grupo2.size > 0 else 0

        nuevo_umbral = (media1 + media2) / 2  # nuevo umbral promedio de medias

        if abs(umbral - nuevo_umbral) < 0.5:
            break
        umbral = nuevo_umbral
    return umbral

imagen = cv2.imread('imagenumbra.jpeg', 0)
if imagen is None:
    raise FileNotFoundError('No hay imagen papito')

umbral_isodata = isodata(imagen)
_, imagen_umbralizada = cv2.threshold(imagen, umbral_isodata, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(imagen, cmap='gray')
plt.title('Imagen inicial')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(imagen_umbralizada, cmap='gray')
plt.title(f'Imagen umbralizada (T={umbral_isodata:.2f})')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def variable_umbral(image, block_size):
    height, width = image.shape
    umbralizada = np.zeros_like(image)

    for i in range(0, height, block_size):
        for j in range(0, width, block_size):
            block = image[i:i+block_size, j:j+block_size]
            umbral = dos_picos(block)  # Usamos la función de dos picos para cada bloque
            _, block_umbralizada = cv2.threshold(block, umbral, 255, cv2.THRESH_BINARY)
            umbralizada[i:i+block_size, j:j+block_size] = block_umbralizada

    return umbralizada